# 03c The attention network, fitted jointly with last year's volatility

`02b-Joint-ridge.ipynb` found that fitting last year's volatility $x$ and the text *together* (with $x$ not
penalised) cuts the ridge's MSE against the straight line from −1.0% to −3.8% on the paper window and from 0.0% to
−2.3% on the strict one, where the two-step fit of 02 and 03 (line first, text on its residual) could not correct
the line's slope. This notebook gives the attention network of 03 the same change, and redoes the second encoder's
ridge (04) the same way.

**The joint network** (`joint.make_attention`) is the network of 03, unchanged, plus one parameter $\gamma$:
$$\hat y_{\text{within}} = f(\text{chunks}) + \gamma\, x_{\text{within}},$$
where "within" means each year's own mean removed. $\gamma$ starts at the no-text within-year slope and is left
out of weight decay, as the joint ridge leaves $x$ unpenalised, so the network can move the weight on $x$ while it
learns the text. Everything else is as in 03 and 03b: the same settings (`attention.py`), early stopping on the
last training year, then a refit on the whole window for the chosen number of epochs, the prediction centred on the
test year (so the level is exactly the line's), scored with `scoring.py`. Both windows, five seeds.

**The second encoder's ridge**: the joint and two-step within-year ridge on the averaged chunk vectors of Yang et
al.'s FinBERT (CLS and token average), from the shards 04 saved on Google Drive.

**This needs a Colab GPU** (Runtime, Change runtime type, T4) for the network, about 30 minutes. The Yang ridge needs
no GPU, only the Drive shards, and takes a few minutes. Anywhere else the notebook only runs the checks.

On Colab, the first cell connects Google Drive, gets the repo and copies `data.pkl`. Locally it does nothing.

In [1]:
import importlib.metadata, importlib.util, os, shutil, subprocess, sys
from pathlib import Path

NOTEBOOK_DIR = "Alex"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    # data.pkl was written with pandas 3 and Colab ships pandas 2, which cannot read it. Install pandas 3 before
    # anything imports pandas. If Colab has already loaded pandas, the runtime has to restart: Run all again.
    if int(importlib.metadata.version("pandas").split(".")[0]) < 3:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas==3.0.6"], check=True)
        if "pandas" in sys.modules:
            print("pandas 3 installed but the old one was already loaded: restarting the runtime. Run all again.")
            os.kill(os.getpid(), 9)
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/dst-sprint2")
    if repo.exists():
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/Oscar339/dst-sprint2.git", str(repo)], check=True)
    processed = repo / "data" / "processed"
    processed.mkdir(parents=True, exist_ok=True)
    if not (processed / "data.pkl").exists():
        shutil.copy(Path("/content/drive/MyDrive/sprint2-data/data.pkl"), processed / "data.pkl")
    os.chdir(repo / NOTEBOOK_DIR)

Mounted at /content/drive


In [2]:
import pickle, platform, time
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import torch

import scoring
import attention
import joint

DEVICE = attention.DEVICE
print(f"Python {platform.python_version()} | torch {torch.__version__} | device {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else " (checks only; train on a Colab GPU)"))
with open("../data/processed/data.pkl", "rb") as f:   # Oscar's hand-over file (Oscar/make_handover.py)
    d = pickle.load(f)
df = scoring.frame(d)
CHUNKS = {"Item 7": d["X_item7"]}                       # per report: an (n_chunks x 768) float16 array
del d
x_all, yr_all = df["logvol_before"].to_numpy(), df["year"].to_numpy()

WINDOWS = ["strict", "paper"]
DRIVE = Path("/content/drive/MyDrive/sprint2-data")
OUT = DRIVE / "attention" if IN_COLAB else Path("../data/output")
RESULTS_FILE, RUNS_FILE = OUT / "alex-03c-joint-attention-results.csv", OUT / "alex-03c-joint-attention-runs.csv"
YANG_DIR, YANG_FILE = DRIVE / "second-encoder" / "yang-item7", OUT / "alex-03c-yang-joint-ridge-results.csv"
print(f"{len(df):,} reports; settings from attention.py: {attention.HP}")

Python 3.13.16 | torch 2.14.1+cu130 | device cuda (Tesla T4)
27,142 reports; settings from attention.py: {'width': 128, 'attention': 64, 'head': 64, 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.01, 'batch': 64, 'max_epochs': 30, 'patience': 5, 'seeds': [0, 1, 2, 3, 4]}


## 1. Checks (run anywhere)

One split (strict window, test year 2006, seed 0) with the epoch limit cut to two, so it runs in about a minute on
a CPU: both versions give finite predictions with exactly the line's level, $\gamma$ starts at the no-text slope
(about 0.87, as in 02b), and the refit trains for the epoch count the first fit chose.

In [3]:
hp_check = {**attention.HP, "max_epochs": 2, "patience": 1}
log = []
preds = joint.make_attention(df, CHUNKS, seed=0, hp=hp_check, log=log)
train, test = scoring.masks(df, 2006, "strict")
a, b = scoring.fit_line(df, train)
for version, predict in preds.items():
    p = predict(df, train, test)
    assert np.isfinite(p).all() and abs(p.mean() - np.mean(a + b * x_all[test])) < 1e-9, version
row = log[0]
assert len(log) == 1                                    # both versions came from one training run
print(f"OK: start slope {row['gamma0_no refit']:.3f}, after training {row['gamma_no refit']:.3f} (no refit) and "
      f"{row['gamma_refit']:.3f} (refit, {row['best_epoch'] + 1} epochs); line slope {row['line_slope']:.3f}; "
      f"{row['seconds']:.0f}s on {DEVICE}")

OK: start slope 0.884, after training 0.847 (no refit) and 0.808 (refit, 2 epochs); line slope 0.872; 22s on cuda


## 2. The joint network (Colab GPU only)

Results are appended to Google Drive after each seed, so if the session dies, running the cell again skips the
seeds already done. At the end the files are copied into `data/output/` in the Colab copy of the repo; download them
from Drive into `data/output/` to commit.

In [4]:
RUN_FULL = IN_COLAB and DEVICE == "cuda"
if RUN_FULL:
    OUT.mkdir(parents=True, exist_ok=True)
    done = pd.read_csv(RESULTS_FILE)["model"].unique().tolist() if RESULTS_FILE.exists() else []
    t_start = time.perf_counter()
    for seed in attention.HP["seeds"]:
        names = {v: f"joint attention, Item 7, {v}, seed {seed}" for v in ["no refit", "refit"]}
        if all(n in done for n in names.values()):
            continue
        print(f"seed {seed}   ({(time.perf_counter() - t_start) / 60:.0f} min so far)")
        log = []
        preds = joint.make_attention(df, CHUNKS, seed=seed, log=log)
        res = pd.concat([scoring.evaluate(df, preds[v], name=n, windows=WINDOWS) for v, n in names.items()],
                        ignore_index=True)
        res.to_csv(RESULTS_FILE, mode="a", header=not RESULTS_FILE.exists(), index=False)
        pd.DataFrame(log).to_csv(RUNS_FILE, mode="a", header=not RUNS_FILE.exists(), index=False)
    for f in [RESULTS_FILE, RUNS_FILE]:
        shutil.copy(f, Path("../data/output") / f.name)
    print(f"done in {(time.perf_counter() - t_start) / 60:.0f} min: "
          f"{pd.read_csv(RESULTS_FILE)['model'].nunique()} version-seed runs, copied to data/output/")
else:
    print("RUN_FULL is False: nothing trained. On a Colab GPU it is True and this cell runs everything.")

seed 0   (0 min so far)
seed 1   (7 min so far)
seed 2   (14 min so far)
seed 3   (21 min so far)
seed 4   (27 min so far)
done in 35 min: 10 version-seed runs, copied to data/output/


## 3. The second encoder's ridge, joint and two-step (Colab, no GPU needed)

The shards from `04-Second-encoder.ipynb` (Yang et al.'s FinBERT, Item 7, 250 reports each) are read from Drive and
put back in `data.pkl`'s row order; each report's chunk vectors are averaged, as in 04.

In [5]:
shards = sorted(YANG_DIR.glob("[12]*_[0-9][0-9][0-9].npz")) if IN_COLAB and YANG_DIR.exists() else []
if shards:
    t0 = time.perf_counter()
    per_key = {}
    for p in shards:
        z = np.load(p)
        cls_all, mean_all, ns = z["cls"], z["mean"], z["n"]
        starts = np.concatenate([[0], np.cumsum(ns)[:-1]])
        for k, s0, n in zip(z["keys"], starts, ns):
            per_key[str(k)] = (cls_all[s0:s0 + n], mean_all[s0:s0 + n])
    assert set(per_key) == set(df["key"]), "every report in data.pkl must have Yang vectors"
    rows = []
    for i, v in [(0, "cls"), (1, "mean")]:
        X = np.stack([per_key[k][i].astype(np.float32).mean(0) for k in df["key"]])
        for kind in ["two-step", "joint"]:
            rows.append(scoring.evaluate(df, joint.make_ridge(df, X, kind), name=f"{kind} within-year ridge, Item 7, Yang {v}"))
    yang = pd.concat(rows, ignore_index=True)
    OUT.mkdir(parents=True, exist_ok=True)
    yang.to_csv(YANG_FILE, index=False)
    shutil.copy(YANG_FILE, Path("../data/output") / YANG_FILE.name)
    print(f"{len(shards)} shards, {yang['model'].nunique()} models in {(time.perf_counter() - t0) / 60:.0f} min; "
          f"saved {YANG_FILE.name}")
    old = pd.read_csv("../data/output/alex-04-ridge-results.csv")
    check = yang[yang.model.str.startswith("two-step")].assign(model=lambda t: t.model.str.replace("two-step ", ""))
    gap = check.merge(old, on=["model", "window", "test_year"])
    print(f"two-step vs 04's ridge: {len(gap)} rows matched, largest MSE difference {(gap.mse_x - gap.mse_y).abs().max():.1e}")
else:
    print("No Yang shards found (they are on Google Drive, from 04): section 3 skipped.")

114 shards, 4 models in 3 min; saved alex-03c-yang-joint-ridge-results.csv
two-step vs 04's ridge: 24 rows matched, largest MSE difference 9.2e-08


## 4. Results

Read from the saved files, so this section runs anywhere once they are in `data/output/`. The change in MSE against
the straight line, averaged over the six test years and over seeds; every level is the line's, so it is all spread.

In [6]:
def avg(d):
    return d.groupby(["model", "window"], sort=False)["change_pct"].mean().unstack().round(2)

rows = []
path = Path("../data/output") / RESULTS_FILE.name
if path.exists():
    r = pd.read_csv(path)
    r["model"] = r["model"].str.replace(r", seed \d+", "", regex=True)
    rows.append(avg(r))
    runs = pd.read_csv(Path("../data/output") / RUNS_FILE.name)
    print(f"{len(runs)} runs; slope on x: line {runs['line_slope'].mean():.3f}, start {runs['gamma0_refit'].mean():.3f}, "
          f"learnt {runs['gamma_refit'].mean():.3f} (refit); mean best epoch {runs['best_epoch'].mean() + 1:.1f}; "
          f"{runs['seconds'].sum() / 60:.0f} GPU-minutes")
ridge = pd.read_csv("../data/output/alex-02b-joint-ridge-results.csv")
rows.append(avg(ridge[ridge.model.isin(["joint within-year ridge, Item 7", "two-step within-year ridge, Item 7"])]))
att = pd.read_csv("../data/output/alex-03b-refit-results.csv")
att = att[att.model.str.contains(", refit, seed")].assign(model="two-step attention, Item 7, refit (03b)")
rows.append(avg(att))
ypath = Path("../data/output") / YANG_FILE.name
if ypath.exists():
    rows.append(avg(pd.read_csv(ypath)))
display(pd.concat(rows))
if path.exists():
    display(r.pivot_table(index=["model", "window"], columns="test_year", values="change_pct", sort=False).round(2))

60 runs; slope on x: line 0.873, start 0.876, learnt 0.767 (refit); mean best epoch 6.9; 35 GPU-minutes


window,strict,paper
model,,
"joint attention, Item 7, no refit",-1.77,-3.14
"joint attention, Item 7, refit",-1.86,-3.87
"two-step within-year ridge, Item 7",-0.03,-1.02
"joint within-year ridge, Item 7",-2.25,-3.82
"two-step attention, Item 7, refit (03b)",-0.52,-1.22
"two-step within-year ridge, Item 7, Yang cls",0.19,-1.08
"joint within-year ridge, Item 7, Yang cls",-2.62,-4.54
"two-step within-year ridge, Item 7, Yang mean",0.42,-0.87
"joint within-year ridge, Item 7, Yang mean",-2.03,-4.03


test_year                                 2001  2002  2003  2004  2005  2006
model                             window                                    
joint attention, Item 7, no refit strict -2.48 -4.93  0.32  1.47 -2.47 -2.54
                                  paper  -1.48 -6.96 -0.90 -2.98 -4.48 -2.04
joint attention, Item 7, refit    strict -2.77 -5.63  0.08  2.36 -4.02 -1.20
                                  paper  -1.60 -7.92 -1.35 -4.40 -4.66 -3.30